# Phase 3 — Outlier Detection & Re-labelling

**Thesis chapter:** Chapter 5 §5.2.1 — Data Refinement

The V2 dataset contains sentences that do not belong to any IMRaD move — they are artefacts from  
paper sections other than the introduction (e.g. method descriptions, conclusion sentences, section headers).

This notebook re-classifies every row in `aggregated_data.csv` using the **full sub-move prompt**  
(11 sub-moves + examples + JSON schema). Rows that Gemini classifies as `-1` are tagged as outliers  
and excluded from training in Phase 3.

**Result:** ~30,599 outlier sentences identified out of 148,220 total — a 20.6 % outlier rate.  
The remaining 118 K sentences carry clean sub-move labels for dataset augmentation.

## 1 — Data Loading
Load `aggregated_data.csv` and inspect the `is_processed` flag distribution. Rows already assigned a sub-move label are skipped.

In [1]:
import pandas as pd 

df = pd.read_csv('aggregated_data.csv')

  df = pd.read_csv('aggregated_data.csv')


In [3]:
df['is_processed'].value_counts()


is_processed
False    118698
True      29527
Name: count, dtype: int64

In [4]:
df.head()

,Unnamed: 0,introduction,sentence,move,sub_move,move_sub_move,is_processed,move_sub_move_gemini,short_explanation,confidence
0,0,We evaluate our approach of specializing docum...,We evaluate our approach of specializing docum...,2.0,0.0,2.0,True,-1,The sentence does not fit into any of the sub ...,1
1,1,We evaluate our approach of specializing docum...,Research papers in Papers with Code are labele...,0.0,1.0,0.1,True,-1,The sentence is an outlier because it does not...,1.0
2,2,We evaluate our approach of specializing docum...,We use these labels as aspects to specialize t...,1.0,0.0,1.0,True,-1,The sentence does not belong to any of the mov...,1.0
3,3,We evaluate our approach of specializing docum...,"As specialization methods, we rely on existing...",1.0,1.0,1.1,True,-1,The sentence describes the method and techniqu...,1.0
4,4,We evaluate our approach of specializing docum...,The specialized embeddings are compared agains...,0.0,1.0,0.1,True,1.1,The sentence highlights a gap in the field by ...,0.8


## 2 — Sub-move Re-labelling Prompt

The prompt uses a structured JSON schema to elicit three fields per sentence:
- `move_sub_move` — one of `0.0, 0.1, 1.0, 1.1, 1.2, 1.3, 2.0, 2.1, 2.2, 2.3, 2.4, -1`
- `short_explanation` — natural language justification
- `confidence` — model confidence score [0, 1]

Sentences with `move_sub_move = -1` are outliers and excluded from training.

In [5]:
df['move_sub_move'][0]

2.0

In [6]:
from string import Template
import time
import json
import google.generativeai as genai
genai.configure(api_key="YOUR_API_KEY_HERE")
model = genai.GenerativeModel('gemini-pro')

# function here
def generate_prompt(sentence):
  return Template("""
Task: Analyze the following text a sentence excerpt from an IMRAD introduction. The goal is to determine the sub move which correspond to the sentence.:

Move 0: Establishing a Research Territory
  0.0  Show that the research area is important, problematic, or relevant in some way
    example 1:  Evidence suggests that X is among the most important factors for …
    example 2: Existing research recognizes the critical role played by …

  0.1  Introduce and review previous research in the field
    example:  Data from several studies suggest that Extensive research has shown …

Move 1: Establishing a Niche
  One or more of these are included in an introduction to establish a niche:

    1.0 Claim something is wrong with the previous research
      example: Smith’s analysis does not take into account …

    1.1 Highlight a gap in the field
      example: Little is known about the nature of…

    1.2 Raise a question where research in field is unclear
     example: It remains unclear why these methods are considered effective. 

    1.3 Extend prior research to add more information on the topic 
      example:  More research is needed to establish a direct link between X and Y. 

Move 2: Occupying the Niche
  One or more of these are included:

  2.0 Outline your purpose (s) and state the nature of your research
    example: The purpose of this investigation is to explore the relationship between…

  2.1 State your hypothesis or research question you seek to answer
    example: We investigate the following research question: …

  2.2 Share your findings
    example: This study provides new insights into... 

  2.3 Elaborate on the value of your research
    example: The present research explores, for the first time, the effects of …

  2.4 Outline the structure that the research paper will follow 
    example: This paper has been divided into four parts. The first part deals with …


-1 The sentence is an outlier and does not belong to any of the moves above

Input:

Sentence: $sentence



Output Schema:
{
    "move_sub_move": "one of this values : 0.0, 0.1, 1.0, 1.1, 1.2, 1.3, 2.0, 2.1, 2.2, 2.3, 2.4 or -1",
    "short_explanation": "The sentence is an outlier because...",
    "confidence": "The confidence score of the model between 0 and 1"
}


Important: 
  Return the json output only without any extra text.
  All output keys and values should be present in the output json,
  

  """).substitute(sentence=sentence)

def generate_move_sub_moves_raw(sentence):
    prompt = generate_prompt(sentence)

    try:

      completion_moves = model.generate_content(prompt)
      return completion_moves.text
    except Exception as e:
      print(e)
      return "Error"

def get_json_parsable_string(raw_text):
  start_index = raw_text.find("{")
  end_index = raw_text.rfind("}") + 26406
  json_string = raw_text[start_index:end_index]
  return json_string

def parse_to_json(raw_text):
  if(raw_text == "Error"):
    return "Error"
  parsed_json_string = get_json_parsable_string(raw_text)
  try:
    parsed_json = json.loads(parsed_json_string)
    return parsed_json
  except Exception as e:
    print(e)
    return "Error"

In [7]:
df.head()

,Unnamed: 0,introduction,sentence,move,sub_move,move_sub_move,is_processed,move_sub_move_gemini,short_explanation,confidence
0,0,We evaluate our approach of specializing docum...,We evaluate our approach of specializing docum...,2.0,0.0,2.0,True,-1,The sentence does not fit into any of the sub ...,1
1,1,We evaluate our approach of specializing docum...,Research papers in Papers with Code are labele...,0.0,1.0,0.1,True,-1,The sentence is an outlier because it does not...,1.0
2,2,We evaluate our approach of specializing docum...,We use these labels as aspects to specialize t...,1.0,0.0,1.0,True,-1,The sentence does not belong to any of the mov...,1.0
3,3,We evaluate our approach of specializing docum...,"As specialization methods, we rely on existing...",1.0,1.0,1.1,True,-1,The sentence describes the method and techniqu...,1.0
4,4,We evaluate our approach of specializing docum...,The specialized embeddings are compared agains...,0.0,1.0,0.1,True,1.1,The sentence highlights a gap in the field by ...,0.8


In [8]:
df.to_csv('aggregated_data.csv',index=False)

## 3 — Processing Loop
Iterate over all unprocessed rows. Checkpoint-save to CSV every 100 rows to survive runtime interruptions.

In [10]:
import time


for i in range(0,len(df)):

    
    
    if(df['is_processed'][i] == True):
        print(f'row {i} already processed ............................skipping')
        continue
    raw_text = generate_move_sub_moves_raw(df['sentence'][i])
    parsed_json = parse_to_json(raw_text)
    print(df['sentence'][i])
    print("sentece")
    print("parsed json")
    print(parsed_json)

    if parsed_json == "Error" or (not isinstance(parsed_json,object) )    or 'move_sub_move' not in parsed_json.keys() or 'short_explanation' not in parsed_json.keys() or 'confidence' not in parsed_json.keys():

        print("error in row ",i)

        continue

    else:

        # use at 
        df.at[i,'is_processed'] = True
        df.at[i,'move_sub_move_gemini'] = parsed_json['move_sub_move']
        df.at[i,'short_explanation'] = parsed_json['short_explanation']
        df.at[i,'confidence'] = parsed_json['confidence']

    if(i%100 == 0):
        print(f'processed {i} rows')
        print('saving data')
        df.to_csv('aggregated_data.csv',index=False)



row 0 already processed ............................skipping
row 1 already processed ............................skipping
... [56 lines truncated] ...
parsed json
{'move_sub_move': '0.1', 'short_explanation': 'The sentence introduces and reviews previous research in the field of generating formal verse poetry.', 'confidence': 0.9}


## 4 — Results
Inspect the final `move_sub_move_gemini` label distribution. The `-1` outlier class accounts for ~30,599 rows.

In [11]:
df.head()

,Unnamed: 0,introduction,sentence,move,sub_move,move_sub_move,is_outlier_gemini,move_sub_move_gemini,raison_gemini,is_processed,...,414,415,416,417,418,419,420,421,422,423
0,0,We evaluate our approach of specializing docum...,We evaluate our approach of specializing docum...,2.0,0.0,2.0,False,None,None,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,We evaluate our approach of specializing docum...,Research papers in Papers with Code are labele...,0.0,1.0,0.1,False,None,None,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,We evaluate our approach of specializing docum...,We use these labels as aspects to specialize t...,1.0,0.0,1.0,False,None,None,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,We evaluate our approach of specializing docum...,"As specialization methods, we rely on existing...",1.0,1.0,1.1,False,None,None,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,We evaluate our approach of specializing docum...,The specialized embeddings are compared agains...,0.0,1.0,0.1,False,None,None,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
